In [1]:
import re
from pathlib import Path
from typing import Dict, List, Tuple
import ast
from itertools import chain
import pandas as pd

In [2]:
def collapse_to_cell_type(gene_set_name: str) -> str:
    """
    Collapse MSigDB-style cell type gene set names to core cell type,
    then normalize casing (smart TitleCase with underscores, preserving acronyms).
    """
    original = gene_set_name.strip()
    name = original
    # 1) Remove everything up to and including _C<number>_
    name2 = re.sub(r"^.*?_C\d+_", "", name)
    # 2) If no C<number>, remove first two underscore-delimited tokens
    #    (keep your existing behavior, but corrected to compare against `name`)
    if name2 == name:
        parts = name.split("_")
        if len(parts) > 2:
            name2 = "_".join(parts[2:])
        else:
            name2 = name
    name = name2
    # 3) Remove trailing cluster indices (_1, _2, etc.)
    name = re.sub(r"_\d+$", "", name)
    # 4) Remove trailing _CELL or _CELLS
    name = re.sub(r"_CELLS?$", "", name)
    # 5) Clean up accidental double underscores
    name = re.sub(r"__+", "_", name).strip("_")
    return name

In [3]:
def read_gmt(gmt_path: str | Path) -> pd.DataFrame:
    """
    Read a GMT file into a DataFrame with columns:
      - gene_set
      - description
      - genes (list[str])
      - n_genes
      - lineno
    GMT format: <gene_set_name>\\t<description>\\t<gene1>\\t<gene2>...
    """
    gmt_path = Path(gmt_path)
    rows: List[Dict] = []

    with gmt_path.open("r", encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.rstrip("\n")
            if not line.strip():
                continue

            parts = line.split("\t")
            if len(parts) < 3:
                # malformed line (no genes)
                continue

            gene_set = parts[0].strip()
            description = parts[1].strip()
            genes = [g.strip() for g in parts[2:] if g and g.strip()]

            # De-duplicate genes within a set while preserving order
            seen = set()
            genes_unique = []
            for g in genes:
                if g not in seen:
                    genes_unique.append(g)
                    seen.add(g)

            rows.append(
                {
                    "gene_set": gene_set,
                    "description": description,
                    "genes": genes_unique,
                    "n_genes": len(genes_unique),
                    "lineno": lineno,
                }
            )

    return pd.DataFrame(rows)

In [4]:
def merge_by_cell_type(gmt_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - merged_df: one row per cell_type with merged/union gene list + summary stats
      - long_df: exploded long-form (cell_type, gene_set, gene) for auditing
    """
    df = gmt_df.copy()
    df["cell_type"] = df["gene_set"].map(collapse_to_cell_type)

    # Long-form for QC / debugging
    long_df = (
        df[["cell_type", "gene_set", "genes"]]
        .explode("genes")
        .rename(columns={"genes": "gene"})
        .dropna(subset=["gene"])
        .reset_index(drop=True)
    )

    # Union of genes per cell type (preserve order, de-dupe)
    def union_genes(list_of_lists: List[List[str]]) -> List[str]:
        seen = set()
        out: List[str] = []
        for genes in list_of_lists:
            for g in genes:
                if g not in seen:
                    out.append(g)
                    seen.add(g)
        return out

    merged_df = (
        df.groupby("cell_type", as_index=False)
        .agg(
            gene_sets=("gene_set", list),
            n_gene_sets=("gene_set", "size"),
            merged_genes=("genes", union_genes),
        )
    )
    merged_df["n_merged_genes"] = merged_df["merged_genes"].map(len)

    # Optional: sort for convenience
    merged_df = merged_df.sort_values(
        ["n_gene_sets", "n_merged_genes"],
        ascending=False
    ).reset_index(drop=True)

    return merged_df, long_df

In [5]:
gmt_path = "gsea_gene_set/c8.all.v2025.1.Hs.symbols.gmt"  # <-- change this to your .gmt path

gmt_df = read_gmt(gmt_path)
merged_df, long_df = merge_by_cell_type(gmt_df)

print(f"Raw gene sets: {len(gmt_df)}")
print(f"Merged cell types: {merged_df.shape[0]}")

merged_df.head(30)

Raw gene sets: 866
Merged cell types: 796


,cell_type,gene_sets,n_gene_sets,merged_genes,n_merged_genes
0,NK_NKT,"[AIZARANI_LIVER_C12_NK_NKT_CELLS_4, AIZARANI_L...",6,"[ALOX5AP, BAG3, BTG2, BTN3A1, CACYBP, CCL5, CD...",431
1,ENDOTHELIAL,"[CUI_DEVELOPING_HEART_C4_ENDOTHELIAL_CELL, JON...",5,"[ACKR3, ADAMTS9, ADGRF5, ADGRG6, AFAP1L1, ALDH...",826
2,KUPFFER,"[AIZARANI_LIVER_C23_KUPFFER_CELLS_3, AIZARANI_...",5,"[ACTB, ADA2, AGFG1, AHR, AIF1, ALDOA, ALOX5, A...",505
3,HEPATOCYTES,"[AIZARANI_LIVER_C11_HEPATOCYTES_1, AIZARANI_LI...",4,"[A1BG, A1CF, AADAC, ABAT, ABCA1, ACAA2, ACADM,...",694
4,EPCAM_POS_BILE_DUCT,[AIZARANI_LIVER_C24_EPCAM_POS_BILE_DUCT_CELLS_...,4,"[ABHD2, ABLIM1, ACTG1, ACTN4, AFDN, AGR2, AKR1...",424
5,GOBLET,"[BUSSLINGER_DUODENAL_GOBLET_CELLS, GAO_LARGE_I...",4,"[AGR2, BCAS1, CLCA1, CREB3L1, EGR1, FCGBP, HPC...",231
6,MACROPHAGE,"[CUI_DEVELOPING_HEART_C8_MACROPHAGE, JONES_OVA...",3,"[A2M, ACTR2, ADA2, ADAP2, ADCY7, AIF1, ALOX5, ...",650
7,THIN_ASCENDING_LIMB,"[LAKE_ADULT_KIDNEY_C10_THIN_ASCENDING_LIMB, LA...",3,"[ACADVL, ACSL4, ACTG1, ACTR2, ADAM10, ADAMTS1,...",522
8,LSECS,"[AIZARANI_LIVER_C13_LSECS_2, AIZARANI_LIVER_C2...",3,"[A2M, ACP5, ACTN1, ADAMTS4, ADD3, ADGRF5, ADGR...",483
9,MVECS,"[AIZARANI_LIVER_C10_MVECS_1, AIZARANI_LIVER_C2...",3,"[A2M, ACKR1, ACP5, ACVRL1, ADAM15, ADAMTS1, AD...",442


In [6]:
# --- Remove organ tokens from merged_df["cell_type"] and re-merge gene sets/genes ---
def _ensure_list(x):
    """Coerce a column entry into a Python list (handles already-list or stringified list)."""
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else [v]
        except Exception:
            # fallback: treat as single string element
            return [x]
    try:
        return list(x)
    except Exception:
        return [x]

# 1) Make sure list-like columns are actually lists
tmp = merged_df.copy()
tmp["gene_sets_list"] = tmp["gene_sets"].apply(_ensure_list)
tmp["merged_genes_list"] = tmp["merged_genes"].apply(_ensure_list)

# 2) Infer likely organ prefixes directly from the underlying gene set names
#    (we use tokens after AUTHOR_; also try common two-token organs like CORD_BLOOD, BONE_MARROW, etc.)
FOLLOWER_TOKENS = {
    "BLOOD","MARROW","CORD","PFC","CORTEX","BRAIN","MUSCLE","INTESTINE","COLON","SKIN","KIDNEY",
    "LUNG","LIVER","HEART","OVARY","PANCREAS","PLACENTA","SPLEEN","THYMUS","TONSIL","ADIPOSE"
}

all_gene_sets = list(chain.from_iterable(tmp["gene_sets_list"].tolist()))

organs_1 = set()
organs_2 = set()
for gs in all_gene_sets:
    if not isinstance(gs, str):
        continue
    toks = gs.strip().split("_")
    if len(toks) >= 2:
        organs_1.add(toks[1])
    if len(toks) >= 3:
        # capture frequent/likely multi-token organs (AUTHOR_CORD_BLOOD_..., AUTHOR_BONE_MARROW_..., etc.)
        if toks[2] in FOLLOWER_TOKENS or toks[1] in FOLLOWER_TOKENS:
            organs_2.add(f"{toks[1]}_{toks[2]}")

# 3) Strip organ prefix from the merged cell_type (best-effort, using inferred organs)
def strip_organ_prefix(cell_type: str) -> str:
    if cell_type is None:
        return cell_type
    s = str(cell_type).strip()
    if not s:
        return s

    parts = s.split("_")
    if len(parts) >= 2 and f"{parts[0]}_{parts[1]}" in organs_2:
        stripped = "_".join(parts[2:])
        return stripped if stripped else s

    if parts[0] in organs_1:
        stripped = "_".join(parts[1:])
        return stripped if stripped else s

    return s

tmp["cell_type_no_organ"] = tmp["cell_type"].apply(strip_organ_prefix)

# 4) Re-merge across the new organ-stripped cell types
def _union_sorted(list_of_lists):
    # preserves uniqueness; sorts for stable viewing
    return sorted(set(chain.from_iterable(list_of_lists)))

merged_df_no_organ = (
    tmp.groupby("cell_type_no_organ", as_index=False)
       .agg(
            gene_sets=("gene_sets_list", _union_sorted),
            merged_genes=("merged_genes_list", _union_sorted),
        )
)

merged_df_no_organ["n_gene_sets"] = merged_df_no_organ["gene_sets"].apply(len)
merged_df_no_organ["n_merged_genes"] = merged_df_no_organ["merged_genes"].apply(len)

# Optional: sort biggest-first for quick inspection
merged_df_no_organ = merged_df_no_organ.sort_values(
    ["n_gene_sets", "n_merged_genes"], ascending=False
).reset_index(drop=True)


print(len(merged_df_no_organ))
merged_df_no_organ.head(20)


723


,cell_type_no_organ,gene_sets,merged_genes,n_gene_sets,n_merged_genes
0,ENDOTHELIAL,"[CUI_DEVELOPING_HEART_C4_ENDOTHELIAL_CELL, DES...","[A2M, A4GALT, ABCA4, ABCG1, ABHD17A, ABI3, ABL...",7,927
1,STROMAL,"[DESCARTES_FETAL_EYE_STROMAL_CELLS, DESCARTES_...","[AAMDC, ABCA10, ABCA6, ABCA8, ABCA9, ABLIM1, A...",7,469
2,NK_NKT,"[AIZARANI_LIVER_C12_NK_NKT_CELLS_4, AIZARANI_L...","[ADA2, ADGRE2, ADGRE5, ADGRG1, AIF1, AKNA, ALO...",6,431
3,GOBLET,"[BUSSLINGER_DUODENAL_GOBLET_CELLS, DESCARTES_F...","[A4GALT, A4GNT, ABCA4, ACSS2, ADGRF1, AGR2, AK...",6,283
4,MESOTHELIAL,"[DESCARTES_FETAL_LIVER_MESOTHELIAL_CELLS, DESC...","[AADAC, ABCA1, ABCG4, ABI3BP, ABRACL, ACOXL, A...",5,982
5,KUPFFER,"[AIZARANI_LIVER_C23_KUPFFER_CELLS_3, AIZARANI_...","[ACADVL, ACOT9, ACSM2A, ACTB, ADA2, ADAM28, AD...",5,505
6,LYMPHOID,"[DESCARTES_FETAL_LIVER_LYMPHOID_CELLS, DESCART...","[ADAM19, ADAMTS7P4, AGAP2, AGMAT, AIRE, ALOX5A...",5,361
7,VASCULAR_ENDOTHELIAL,[DESCARTES_FETAL_EYE_VASCULAR_ENDOTHELIAL_CELL...,"[A2M, ABCB1, ABCG2, ACP5, ACSM1, ACSM2B, ACVRL...",5,286
8,MICROGLIA,"[DESCARTES_FETAL_EYE_MICROGLIA, DESCARTES_MAIN...","[ABCA2, ABCE1, ABHD13, ABLIM1, ABR, ACAP3, ACK...",4,1042
9,HEPATOCYTES,"[AIZARANI_LIVER_C11_HEPATOCYTES_1, AIZARANI_LI...","[A1BG, A1CF, A2M, AADAC, AASS, ABAT, ABCA1, AB...",4,694


In [7]:
def write_gmt(df, name_col, genes_col, out_gmt):
    """
    Write a dataframe to GMT format.

    Parameters
    ----------
    df : pd.DataFrame
        One row per gene set
    name_col : str
        Column containing gene set names
    genes_col : str
        Column containing iterable of genes (list/set/tuple)
    out_gmt : str or Path
        Output .gmt file
    """
    out_gmt = Path(out_gmt)

    with out_gmt.open("w") as f:
        for _, row in df.iterrows():
            genes = row[genes_col]

            # handle strings accidentally stored as comma-separated
            if isinstance(genes, str):
                genes = [g for g in genes.split(",") if g]

            genes = sorted(set(genes))
            if len(genes) == 0:
                continue

            line = [
                row[name_col],               # gene set name
                "merged_from_C8",             # description (free text)
                *genes
            ]
            f.write("\t".join(line) + "\n")

    print(f"✅ Wrote {len(df)} gene sets to {out_gmt}")

In [9]:
write_gmt(
    df=merged_df_no_organ,
    name_col="cell_type_no_organ",
    genes_col="merged_genes",
    out_gmt="gsea_gene_set/C8_merged_by_cell_type.gmt"
)

✅ Wrote 723 gene sets to gsea_gene_set/C8_merged_by_cell_type.gmt


In [10]:
merged_df_no_organ.to_csv('data/merged_c8_gene_set.csv', sep='\t')